<center><a href="https://www.nvidia.com/en-us/training/"><img src="../images/nvidia_header.png" width="400"/></a></center>

# 3. RT-VLM: Captioning & Prompting

RT-VLM (captioning with a vision language model) runs Cosmos 3 Super Reasoner on video clips. It splits a clip
into chunks, samples frames from each chunk, and answers your prompt once per chunk. You caption the
`traffic_short` clip from notebook 2 and change the settings that shape the answer.

In this notebook you will:

- Register a clip from VIOS (video storage) with RT-VLM and caption it.
- Change the prompt, the chunk size and the number of sampled frames, and compare the answers.
- Ask for a report, JSON and a one-word answer through the OpenAI-style route.
- Find the stored captions in Elasticsearch (the search index).
- Estimate what continuous captioning costs.

**Time:** 35 minutes

<!-- generated by task1/scripts/course_path.py from part1/course_path.json; edit the JSON, not this cell -->
> **Part 1** &middot; about 35 min &nbsp;|&nbsp; [Home](../00_Home.ipynb)
>
> **Before you start:** `traffic_short` added to VIOS in notebook 2.
>
> Previous: [2. RT-CV: Detection & Tracking](02-rt-cv-detection-and-tracking.ipynb) &middot; Next: [4. RT-Embed: Video Search](04-rt-embed-video-search.ipynb)


<img src="../images/course/map-caption.png" width="960" alt="The VSS pipeline map with VIOS, RT-CV, RT-VLM, Kafka and Elasticsearch lit, and your notebook, the VSS agent service and RT-Embed dimmed. This notebook follows the RT-VLM branch: a clip from VIOS goes in, and the captions travel through Kafka into Elasticsearch.">

## 3.1 Register the clip with RT-VLM

You work with the 16-second `traffic_short` clip that you added to VIOS in notebook 2. The setup cell finds the
clip in VIOS and reads the model id from RT-VLM. If VIOS has no `traffic_short`, the cell stops and
sends you back to notebook 2.

In [ ]:
import base64, json, re, time
import requests
from IPython.display import Markdown, display

import course_setup, labtools
from cleanup import remember             # records what this notebook creates, so it can be removed later

SOURCE = "traffic_short"                 # the file stem: RT-CV's sensorId for this clip
ANCHOR = "2025-01-01T00:00:00.000Z"      # the clip's start on its VIOS timeline; RT-CV indexes it under this date too
CLIP_SECONDS = 16                        # traffic_short.mp4 is 16 s long

found = course_setup.find_sensor(SOURCE)
if found is None:
    raise RuntimeError("VIOS has no source named %r. Run notebook 2 first: it adds the clip to VIOS." % SOURCE)
sensor_id = found["sensorId"]            # the VIOS UUID: VIOS cuts clips by this id
print("file stem:", SOURCE, "| VIOS name:", found["name"], "| VIOS sensor id:", sensor_id)

model = course_setup.vlm_model_id()
print("RT-VLM model (Cosmos 3 Super Reasoner):", model)

Registering the clip takes two calls. `course_setup.register_vios_window` makes both:

1. `GET /vios/vst/api/v1/storage/file/<sensor id>/url` with `startTime`, `endTime` and `container=mp4`. VIOS
   answers with a URL for that window of the video.
2. `POST /rtvi-vlm/v1/files` with that `url`, `purpose=vision`, `media_type=video` and a `creation_time`. RT-VLM
   fetches the video itself and returns a file id.

The `creation_time` is the clip's start on its VIOS timeline. RT-VLM stamps every caption with times counted from
it, so the captions line up with the recording and its detections. The file id also names the Elasticsearch index
that stores this file's captions (section 3.8).

In [ ]:
# Arguments: VIOS sensor id, window start, window end, creation_time for the registered file.
file_id = course_setup.register_vios_window(
    sensor_id, ANCHOR, course_setup.iso_offset(ANCHOR, CLIP_SECONDS), ANCHOR)
remember("caption", file_id)             # puts the RT-VLM file and its caption index on the cleanup list
print("RT-VLM file id:", file_id)

## 3.2 Caption the clip

`chunk_duration` splits the registered window into chunks, and RT-VLM answers once per chunk. Ten seconds gives
two chunks for this 16-second clip:

<img src="../images/course/chunking.png" width="960" alt="A filmstrip of the 16-second traffic_short window, registered with creation_time 2025-01-01T00:00:00.000Z. With chunk_duration=10 it becomes two chunks and two RT-VLM calls, answered for 00:00:00 to 00:00:10 and 00:00:10 to 00:00:16; with chunk_duration=0 it is one chunk, one call and one answer for the whole window.">

`course_setup.caption(file_id, prompt, chunk_duration)` sends `POST /rtvi-vlm/v1/generate_captions`. The request
body holds the file `id`, the `model` id, the `prompt` and `chunk_duration`. The response holds `chunk_responses`,
a list with one entry per chunk. RT-VLM counts the chunk times from the `creation_time` you registered in section
3.1, so they start at 00:00:00.

The helper below prints each caption under a frame from the start of its chunk. It also prints how long the model
took, `vlm_latency_ms`, and how many frames it saw, `frame_count`.

In [ ]:
def ms(chunk):
    """A chunk's vlm_latency_ms, rounded for printing; '?' when the response has no such field."""
    value = chunk.get("vlm_latency_ms")
    return "%.0f" % value if isinstance(value, (int, float)) else "?"


def show_chunks(chunks, width=480):
    """Print each caption under one frame from the start of its chunk. Returns the chunks unchanged."""
    for c in chunks:
        start_s = course_setup.offset_seconds(ANCHOR, c["start_time"])   # seconds from the start of the clip
        frame = labtools.frame_from_mp4(course_setup.clip_bytes(
            sensor_id, course_setup.iso_offset(ANCHOR, start_s),
            course_setup.iso_offset(ANCHOR, start_s + 1)), width=width)
        print("[%s to %s]  vlm_latency_ms: %s  frame_count: %s" % (
            c["start_time"][11:19], c["end_time"][11:19], ms(c), c.get("frame_count", "?")))
        display(frame)
        print(c["content"].strip(), "\n")
    return chunks

In [ ]:
DESCRIBE = "Describe what happens in this traffic scene."

# Two chunks, so two RT-VLM calls.
describe = show_chunks(course_setup.caption(file_id, DESCRIBE, chunk_duration=10))

Check one concrete claim, such as an action, a vehicle's position or the order of events. In this clip, a white
pickup and a red sedan collide at about 3.5 to 4 s. They stay in contact until about 10 s and pull apart at about
10.5 to 11 s. Does the first caption mention the contact? Does the second describe the two vehicles moving apart?
If the first caption misses the contact, look at its `frame_count`. Section 3.4 shows why.

Each picture above is one frame from the start of its chunk, not necessarily a frame the model saw. A caption can
add context that detection records lack, but a caption that reads well can still be wrong.

## 3.3 Change the question

Ask two new questions about the same clip. One asks for a count and one asks about safety. Before you run them,
predict the count and whether each safety answer will be "yes". The safety question uses 10-second chunks, so it
gets two answers, one per chunk.

The count request uses `chunk_duration=0`, which gives one answer for the whole 16 s. You can then compare it with
a detector count over the same window. Look at the shape of each answer as well as its content. The prompt decides
both.

In [ ]:
# One chunk for the whole clip: the same 0 to 16 s window as the detector query below.
count_cars = show_chunks(course_setup.caption(
    file_id, "Count the cars visible in this clip. Answer with only a number.", chunk_duration=0))

In [ ]:
unsafe = show_chunks(course_setup.caption(
    file_id, "Is anything unsafe happening? Answer yes or no, then explain in one sentence.", chunk_duration=10))

Check both answers. Is the count a bare number, as the prompt asked? Does each safety answer start with yes or no,
and does its sentence match what happens in that chunk?

### Optional: compare the counts fairly

A VLM count means little until you compare it with something that counts the same thing. The query below matches
the count request: the same source, the same 0 to 16 s window, and the class `car`. TrafficCamNet RT-DETR has no
"vehicle" class, so the prompt and the query both use `car`.

**Predict first:** how many distinct car tracklets will RT-CV (detection and tracking) report? Notebook 2 counted
the distinct tracklets of every class for the whole clip.

**Your turn:** In the next cell, replace the two `FIXME`s with nested field names. One field holds an object's
class and the other holds its tracklet id. Notebook 2 used both. Then run the check cell. It prints the VLM
count and the RT-CV count side by side. A correct query counts more than zero car tracklets. Click the `...` under the
cell for the solution.

In [ ]:
# TODO
# Same window (0 to 16 s) and same class (car) as the count prompt. Name the two nested fields.
agg = course_setup.es("POST", "/mdx-raw-2025-01-01/_search", json={
    "size": 0,
    "query": {"bool": {"filter": [
        {"term": {"sensorId.keyword": SOURCE}},
        {"range": {"timestamp": {"gte": ANCHOR, "lt": course_setup.iso_offset(ANCHOR, CLIP_SECONDS)}}},
    ]}},
    "aggs": {"o": {"nested": {"path": "objects"}, "aggs": {
        "cars": {"filter": {"term": {FIXME: "car"}}, "aggs": {         # the nested field that holds the class
            "tracklets": {"cardinality": {"field": FIXME}}}}}}},        # the nested field that holds the tracklet id
})
rtcv_cars = agg["aggregations"]["o"]["cars"]["tracklets"]["value"]

In [ ]:
# SOLUTION
# Same window (0 to 16 s) and same class (car) as the count prompt. Name the two nested fields.
agg = course_setup.es("POST", "/mdx-raw-2025-01-01/_search", json={
    "size": 0,
    "query": {"bool": {"filter": [
        {"term": {"sensorId.keyword": SOURCE}},
        {"range": {"timestamp": {"gte": ANCHOR, "lt": course_setup.iso_offset(ANCHOR, CLIP_SECONDS)}}},
    ]}},
    "aggs": {"o": {"nested": {"path": "objects"}, "aggs": {
        "cars": {"filter": {"term": {"objects.type.keyword": "car"}}, "aggs": {   # the class lives on each object
            "tracklets": {"cardinality": {"field": "objects.id.keyword"}}}}}}},    # distinct tracklet ids, not detections
})
rtcv_cars = agg["aggregations"]["o"]["cars"]["tracklets"]["value"]

In [ ]:
# Check: RT-CV returned a number of tracklets and the VLM answered with a number. Then print them side by side.
if not isinstance(rtcv_cars, int) or rtcv_cars < 0:
    raise ValueError("the cardinality aggregation returned %r; the two fields are objects.type.keyword and "
                     "objects.id.keyword" % (rtcv_cars,))
if rtcv_cars == 0:
    raise RuntimeError("RT-CV reports no car tracklets in 0 to 16 s of %r, yet notebook 2 drew them. A field name "
                       "that is not in the index counts zero without an error: check both FIXMEs." % SOURCE)
# The same query with the two right fields. Your count must match it.
reference = course_setup.es("POST", "/mdx-raw-2025-01-01/_search", json={
    "size": 0,
    "query": {"bool": {"filter": [
        {"term": {"sensorId.keyword": SOURCE}},
        {"range": {"timestamp": {"gte": ANCHOR, "lt": course_setup.iso_offset(ANCHOR, CLIP_SECONDS)}}},
    ]}},
    "aggs": {"o": {"nested": {"path": "objects"}, "aggs": {
        "cars": {"filter": {"term": {"objects.type.keyword": "car"}}, "aggs": {
            "tracklets": {"cardinality": {"field": "objects.id.keyword"}}}}}}},
})["aggregations"]["o"]["cars"]["tracklets"]["value"]
if rtcv_cars != reference:
    raise ValueError("your query counted %d, but RT-CV has %d distinct car tracklet ids in 0 to 16 s. Check which "
                     "FIXME names the class field and which names the tracklet id field." % (rtcv_cars, reference))
vlm_text = count_cars[0]["content"].strip()
match = re.search(r"\d+", vlm_text)
if match is None:
    raise ValueError("Cosmos 3 Super Reasoner did not answer with a number; its reply was %r. "
                     "Make the prompt stricter and rerun the count cell in section 3.3." % vlm_text)
vlm_cars = int(match.group())

print("%-12s %-8s %26s %24s" % ("window", "class", "Cosmos 3 Super Reasoner", "TrafficCamNet RT-DETR"))
print("%-12s %-8s %26d %24d" % ("0 to 16 s", "car", vlm_cars, rtcv_cars))
print("PASS: both counts cover 0 to 16 s and the class car.")

Two numbers over the same window and class can still measure different things. The VLM answers from the frames
it sampled, so it can count a car twice or miss one that is small or hidden. Look at `frame_count` in its chunk: at
the default, the model saw one frame of the 16 s, so it could count only the cars in that frame (section 3.4
changes that). RT-CV counts tracklet ids, and a tracklet that breaks and restarts gets a new id, so one car can
count twice. If they agree, that is a good sign. If they
disagree, watch the clip to find out why. Neither number is a measurement until you count the cars in the footage
yourself.

## 3.4 Change the chunk size, the sampled frames and other request fields

Two settings decide how much of the video the model sees: the chunk size and the number of frames sampled from
each chunk.

Smaller chunks mean more requests, and they make it easier to tell when an event happened. Larger chunks give the
model a longer stretch of video for one answer. Run the prompt from section 3.2 again with `chunk_duration=0`, so
the chunk size is the only change.

In [ ]:
# Same prompt as section 3.2. Only chunk_duration changes.
whole = show_chunks(course_setup.caption(file_id, DESCRIBE, chunk_duration=0))

Compare this single answer with the two from section 3.2. Does it keep the order of events? Does it still mention
the contact between the pickup and the sedan? Choose a chunk size by testing it on your own task, not by assuming that
shorter or longer is always better.

### Change the number of sampled frames

RT-VLM does not show the model every frame of a chunk. It samples frames, spaced evenly through the chunk. Two
request fields control how many:

- `num_frames_per_second_or_fixed_frames_chunk` sets the number of frames.
- `use_fps_for_chunking` says how to read that number. With `false` it is frames per chunk. With `true` it is
  frames per second.

When a request sends neither field, RT-VLM uses a default that is set when the service is deployed. Each chunk
response reports `frame_count`, the number of frames the model actually saw. `input_tokens` counts what went into
the model: the frames and the prompt. `course_setup.caption` sends both fields when you pass `frames=`, and
`use_fps=True` switches to frames per second.

The next cell prints `frame_count`, `input_tokens` and `vlm_latency_ms` for the section 3.2 captions, which used
the default.

In [ ]:
def frame_table(chunks, label):
    """One line per chunk: its window, the frames the model saw, the input tokens and vlm_latency_ms."""
    missing = sorted({k for c in chunks for k in ("frame_count", "input_tokens", "vlm_latency_ms") if k not in c})
    if missing:
        raise KeyError("these RT-VLM chunk responses have no %s; fields present: %s" % (missing, sorted(chunks[0])))
    print(label)
    print("  %-22s %12s %13s %15s" % ("window", "frame_count", "input_tokens", "vlm_latency_ms"))
    for c in chunks:
        window = "%s to %s" % (c["start_time"][11:19], c["end_time"][11:19])
        print("  %-22s %12s %13s %15s" % (window, c["frame_count"], c["input_tokens"], ms(c)))


frame_table(describe, "Section 3.2, deployment default:")

In [ ]:
# Same clip, prompt and chunk size as section 3.2. Only the number of sampled frames changes.
dense = course_setup.caption(file_id, DESCRIBE, chunk_duration=10, frames=8, use_fps=False)
frame_table(dense, "8 frames per chunk:")

for before, after in zip(describe, dense):
    print("\n[%s to %s]" % (after["start_time"][11:19], after["end_time"][11:19]))
    print("default :", before["content"].strip())
    print("8 frames:", after["content"].strip())

counts = [c["frame_count"] for c in dense]
if any(n != 8 for n in counts):
    raise RuntimeError("RT-VLM reported frame_count %s, not 8: the frames setting was not applied." % counts)

Compare the two tables first. If the default was lower than 8, the model now sees more of each chunk:
`input_tokens` grows, and `vlm_latency_ms` can grow with it. Then compare the words. Wording can change a little
from run to run, so look for details that only the 8-frame captions mention. If the default `frame_count` was 1,
the model described ten seconds of video from a single picture. The moment the two vehicles collide, at about 3.5
to 4 s, may not be in that picture.

More frames cost more per request. Fewer frames can miss short events.

Look back at `frame_count` for the one-chunk answer earlier in this section. If it matches a 10-second chunk's
count, that answer covered all 16 seconds with no more frames than one 10-second chunk.

### Set a system prompt and a token cap

You can set more fields on each `generate_captions` request. These are the ones you are most likely to use:

| Field | What it does |
|---|---|
| `system_prompt` | Instructions that apply to the whole request, kept separate from the prompt |
| `max_tokens` | The most tokens the answer may use |
| `temperature`, `top_p` | How much the wording varies from run to run |
| `response_format` | `{"type": "json_object"}` asks for a JSON answer |

A system prompt suits rules that hold for every request, such as how long to answer. The
[Cosmos 3 prompt guide](https://github.com/NVIDIA/cosmos/blob/main/cookbooks/cosmos3/reasoner/reasoner_prompt_guide.md)
puts the role, the focus and the format in the prompt itself, which is what section 3.5 does.

`course_setup.caption` accepts `system_prompt=` and `max_tokens=` and sends each one only when you pass it. It does
not take `temperature`, `top_p` or `response_format`. The next cell asks the section 3.2 question again. This time
a system prompt asks for one sentence, and `max_tokens` caps the answer at 60 tokens. Compare `output_tokens` and
`vlm_latency_ms` with section 3.2: shorter answers usually take less time. An answer that stops mid-sentence hit
the cap.

In [ ]:
short = course_setup.caption(
    file_id, DESCRIBE, chunk_duration=10,
    system_prompt="Answer in one sentence. Describe only what is visible.",
    max_tokens=60)

for before, after in zip(describe, short):
    print("[%s to %s]" % (after["start_time"][11:19], after["end_time"][11:19]))
    print("  section 3.2: output_tokens %s, vlm_latency_ms %s" % (before.get("output_tokens", "?"), ms(before)))
    print("  short      : output_tokens %s, vlm_latency_ms %s" % (after.get("output_tokens", "?"), ms(after)))
    print("  " + after["content"].strip(), "\n")

## 3.5 Prompt through the OpenAI-style route

So far you used `generate_captions`, which chunks a registered file and stores the captions. RT-VLM also serves
an OpenAI-style `chat/completions` route. You send a clip and a prompt, and you get back one answer with its finish
reason and token counts. One answer per request makes this route handy for comparing prompts.

The OpenAI client needs only the base URL `http://nginx/rtvi-vlm/v1` and a placeholder key. The service does not
check the key, but the client needs one. The model id comes from `GET /rtvi-vlm/v1/models`, so you never type it.

The clip is the same 16-second window, cut from VIOS and sent inside every request as a base64
`data:video/mp4` URL. The whole clip travels with each request, so this route suits short clips, not whole
recordings.

In [ ]:
from openai import OpenAI

VLM_BASE_URL = "http://nginx/rtvi-vlm/v1"

# max_retries=0: a failed request fails once, with the server's message, instead of being sent again.
client = OpenAI(base_url=VLM_BASE_URL, api_key="EMPTY", timeout=600, max_retries=0)

models = client.models.list().data        # raises openai.APIStatusError with the server's body on an HTTP error
if not models:
    raise RuntimeError(f"{VLM_BASE_URL}/models returned no models: RT-VLM is up but has not loaded a model.")
MODEL = models[0].id
print("RT-VLM model id:", MODEL)

In [ ]:
# The same 16 s window you registered in section 3.1, cut from VIOS.
clip = course_setup.clip_bytes(sensor_id, ANCHOR, course_setup.iso_offset(ANCHOR, CLIP_SECONDS))
if len(clip) < 50_000:
    raise RuntimeError(f"VIOS returned only {len(clip)} bytes for the first {CLIP_SECONDS} s of {SOURCE}; "
                       "compare the window with the timeline that notebook 2 printed.")

# The clip rides inside the JSON body of every request. Base64 makes it about a third larger.
video_data_url = "data:video/mp4;base64," + base64.b64encode(clip).decode("ascii")
print(f"{len(clip) / 1e6:.1f} MB from VIOS, {len(video_data_url) / 1e6:.1f} MB per request body")
labtools.show_video(clip)

Every request below has the same shape. The video goes before the text, the order the model was trained on. The
system message is a plain "You are a helpful assistant." The
[Cosmos 3 prompt guide](https://github.com/NVIDIA/cosmos/blob/main/cookbooks/cosmos3/reasoner/reasoner_prompt_guide.md)
puts the role, the focus and the format in the user message instead.

Two sampling settings travel with each request: `temperature` and `top_p`. Low values keep answers comparable from
run to run. The reasoning call in section 3.7 uses the prompt guide's 0.6 and 0.95. The route also accepts
`frequency_penalty`, but RT-VLM does not use it, so these cells do not send it.

Each request also asks for `FRAMES = 8` frames, with the same two fields as section 3.4. Without them, RT-VLM
samples a single frame on this deployment, and the model answers about all 16 seconds from that one picture. The
OpenAI client has no argument for these fields, so `ask()` adds them to the request body with `extra_body`. Eight
frames over this clip are about 2 seconds apart. For a longer clip, raise `FRAMES` or switch to frames per second,
as section 3.4 explains.

On this route, `prompt_tokens`, the "tokens in" that the next cells print, counts only the text you send, not the
video. Unlike `input_tokens` in section 3.4, it cannot show how many frames the model saw.

In [ ]:
SYSTEM = "You are a helpful assistant."
FRAMES = 8                                       # frames that RT-VLM samples from the clip in each request


def ask(prompt, max_tokens=1024, temperature=0.4, top_p=0.4):
    """One chat/completions request on the clip. Returns the text, finish_reason, seconds and token counts."""
    started = time.time()
    response = client.chat.completions.create(   # raises openai.APIStatusError with the body on 4xx/5xx
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": [
                {"type": "video_url", "video_url": {"url": video_data_url}},
                {"type": "text", "text": prompt},
            ]},
        ],
        max_tokens=max_tokens, temperature=temperature, top_p=top_p,
        # RT-VLM's own fields, as in section 3.4: FRAMES frames over the whole clip. Without them it samples one.
        extra_body={"num_frames_per_second_or_fixed_frames_chunk": FRAMES, "use_fps_for_chunking": False},
    )
    seconds = time.time() - started
    choice = response.choices[0]
    text = choice.message.content or ""
    if not text.strip():
        raise RuntimeError(f"RT-VLM returned an empty answer (finish_reason={choice.finish_reason!r}).")
    usage = response.usage
    return {"text": text, "finish_reason": choice.finish_reason, "seconds": seconds,
            "prompt_tokens": usage.prompt_tokens if usage else None,
            "completion_tokens": usage.completion_tokens if usage else None}

In [ ]:
def final_answer(text):
    """Cosmos 3 Super Reasoner may reason inside <think>...</think> on its own; the answer follows the closing tag."""
    return text.split("</think>", 1)[1].strip() if "</think>" in text else text.strip()


def parse_json_answer(text):
    """Parse a JSON answer, or raise with the raw answer attached."""
    body = final_answer(text)
    fenced = re.search(r"```(?:json)?\s*(.*?)```", body, re.S)   # the model sometimes wraps JSON in a Markdown fence
    if fenced:
        body = fenced.group(1)
    try:
        return json.loads(body)
    except json.JSONDecodeError as err:
        raise ValueError(f"Not valid JSON ({err}). Name the keys, forbid prose and fences, then rerun."
                         f"\n--- raw answer ---\n{text}") from None


def show(result, label):
    """Print one answer with its cost. finish_reason 'length' means the cap was hit and the text is incomplete."""
    note = "" if result["finish_reason"] == "stop" else "  <- hit max_tokens: incomplete"
    print(f"[{label}] {result['seconds']:.1f} s | finish_reason: {result['finish_reason']}{note} | "
          f"tokens in/out: {result['prompt_tokens']}/{result['completion_tokens']}")
    print(result["text"].strip(), "\n")

### Compare a vague prompt with a specific one

A VLM answers the question you ask. A vague prompt gets a vague caption. A prompt that gives the model a role, a
focus and an answer format gets something an operator can act on.

The prompts live in `PROMPT_PRESETS` in the next cell. Run it as it is first. Then change `BAD_PRESET` and
`GOOD_PRESET`, or add your own entry, and rerun the two request cells to compare any pair.

**Predict first:** will the vague prompt mention the collision at all? At what second will the specific prompt
place it?

After running, look for:

- Did the vague prompt produce anything a traffic operator could act on?
- Did the specific prompt follow the numbered structure, and say "not visible" where it should?
- Did either answer describe an event that is not in the clip?

In [ ]:
# Prompt presets: pick two by name, or add your own, then rerun the two request cells below.
PROMPT_PRESETS = {
    "bad_vague": "What's in this video?",

    "good_inspection": """You are a traffic operations monitor reviewing an intersection camera clip.

Watch the video and report:
1. Every vehicle visible, its type (car, truck, bus, motorcycle), and what it is doing
2. Whether any pedestrians or cyclists are present in or near the roadway and whether the crossing is clear (count and location only; do not describe individuals)
3. Any traffic incidents or hazards (stalled vehicles, collisions, near-misses, red-light running, debris or blocked lanes), with the approximate time in the clip at which each occurs

Watch the whole clip to the end before answering: events can happen in the last seconds. Be specific about where in the frame each observation occurs. If something is not visible, say so; do not guess.""",

    "stalled_vehicle_focus": """You are a traffic operations monitor. Focus only on stopped vehicles.
For each vehicle that is not moving, state its lane position and whether it is stopped normally (at a signal, in a queue) or abnormally (in a live lane, on the median, after a crash).
If no vehicle is stopped abnormally, answer exactly: "No stalled vehicle.\"""",

    "collision_focus": """You are a traffic operations monitor. Focus only on collisions and near-misses.
Describe any contact or sudden evasive maneuver between vehicles, with the approximate time in the clip and the lanes involved.
If there is no collision or near-miss, answer exactly: "No collision.\"""",

    # Your own prompt: give it a role, a focus and an answer format, like the presets above.
    "custom": """You are a traffic operations monitor. <write your own instructions here>""",
}

BAD_PRESET = "bad_vague"           # change these two names to compare any pair of presets
GOOD_PRESET = "good_inspection"

bad_prompt = PROMPT_PRESETS[BAD_PRESET]
good_prompt = PROMPT_PRESETS[GOOD_PRESET]
print(f"[{BAD_PRESET}]\n{bad_prompt}\n\n[{GOOD_PRESET}]\n{good_prompt}")

In [ ]:
# The vague prompt: no role, no focus, no format.
# Low temperature and top_p, so the prompt is the only thing that changes between the two runs.
bad = ask(bad_prompt, max_tokens=1024, temperature=0.2, top_p=0.2)
show(bad, BAD_PRESET)

In [ ]:
# The specific prompt: a role, a focus and an answer format. Same sampling as the vague run.
good = ask(good_prompt, max_tokens=1024, temperature=0.2, top_p=0.2)
show(good, GOOD_PRESET)

Compare the two answers. The vague prompt usually gets a one-line description. The specific prompt should get a
report an operator can act on. Now check the report against the clip. Did it catch the collision at about 3.5 to 4 s? A
well-structured answer can still be wrong.

Pick one claim from the report, such as a vehicle's position, a time or a lane. Can you see it in the clip, or did
the model infer it from what it saw?

The figure marks the four parts of the specific prompt: a role, a focus, an answer format and an honesty line. The honesty line tells the model to say when something is not
visible instead of guessing.

<img src="../images/course/prompt-anatomy.png" width="960" alt="On the left, the vague prompt bad_vague, 'What's in this video?', with no role, no focus and no format. On the right, the good_inspection prompt with its parts bracketed as role, format, focus and honesty, above the line 'Structure makes the output usable, not true.'">

## 3.6 Ask for structured output

Free text is for people. The next program in a pipeline, such as a dashboard, an index or an alert rule, needs
output it can parse. You can ask Cosmos 3 Super Reasoner for a report, a JSON object or a one-word answer. What
counts is whether a parser accepts the output, not whether it looks structured.

Start with a Markdown report with fixed sections. The prompt names the sections. Check whether the model kept them,
and what it wrote under "Incidents & Hazards".

In [ ]:
report_prompt = """You are a video analytics report generator. Watch the video and produce a traffic report in Markdown with exactly these sections:

## Summary
One paragraph describing the intersection and overall traffic activity.

## Timeline
A bulleted list of notable events in chronological order.

## Incidents & Hazards
A bulleted list of any traffic incidents or safety concerns. If there are none, state that explicitly."""

report = ask(report_prompt, max_tokens=1024, temperature=0.4, top_p=0.4)
print(f"[report] {report['seconds']:.1f} s | finish_reason: {report['finish_reason']} | "
      f"tokens in/out: {report['prompt_tokens']}/{report['completion_tokens']}")
display(Markdown(final_answer(report["text"])))

Now ask for the events as JSON with timestamps, a shape a pipeline can store and index. `parse_json_answer`
returns the parsed JSON, or raises with the raw answer attached. An answer that does not parse has failed, however
well it reads.

In [ ]:
json_prompt = ("Describe the notable events in the provided video. Provide the result in json format with "
               "'mm:ss.ff' format for time depiction for each event. Use keywords 'start', 'end' and 'caption' "
               "in the json output. Return only the JSON.")
events_answer = ask(json_prompt, max_tokens=2048, temperature=0.4, top_p=0.4)
show(events_answer, "json_events")

parsed = parse_json_answer(events_answer["text"])        # raises with the raw text when it does not parse
events = parsed if isinstance(parsed, list) else (parsed.get("events") if isinstance(parsed, dict) else None)
if not isinstance(events, list):
    raise ValueError(f"Parsed JSON but found no list of events; top level is {type(parsed).__name__}: "
                     f"{sorted(parsed) if isinstance(parsed, dict) else parsed}")
print(f"Parsed {len(events)} event(s):")
for event in events:
    print(f"  {event.get('start')} -> {event.get('end')}: {event.get('caption')}")

**Your turn:** Write a prompt for a use case of your own that returns a JSON object with keys you choose. In the
next cell, replace each `FIXME`. List the key names your code will read, and write a prompt that names each key
with its allowed values. Then run the check cell. It passes when the answer parses and contains every key. Click the `...` under the
cell for the solution.

In [ ]:
# TODO
MY_KEYS = ["FIXME", "FIXME"]   # the key names your code reads, for example "event_present", "evidence"

my_prompt = """You are a FIXME. Watch the whole clip before answering.
Return only a JSON object with exactly these keys: FIXME.
FIXME (one line per key: what it means and the allowed values)
No prose before or after the JSON and no Markdown fence."""

In [ ]:
# SOLUTION
MY_KEYS = ["vehicle_stopped_in_live_lane", "evidence", "confidence"]

my_prompt = """You are a traffic operations monitor. Watch the whole clip before answering.
Return only a JSON object with exactly these keys: "vehicle_stopped_in_live_lane", "evidence", "confidence".
"vehicle_stopped_in_live_lane": true or false.
"evidence": one sentence describing only what is visible that supports the value; if nothing is visible, say so.
"confidence": one of "high", "medium", "low".
No prose before or after the JSON and no Markdown fence."""

In [ ]:
# Check: the answer parses as a JSON object and has every key you named.
if "FIXME" in my_prompt or "FIXME" in MY_KEYS:
    raise ValueError("Replace every FIXME in the exercise cell (or run the solution cell) before checking.")
mine = ask(my_prompt, max_tokens=512, temperature=0.2, top_p=0.2)
show(mine, "your_json")

obj = parse_json_answer(mine["text"])
if not isinstance(obj, dict):
    raise ValueError(f"Expected a JSON object, got {type(obj).__name__}: ask for 'a JSON object', not 'JSON'.")
missing = [key for key in MY_KEYS if key not in obj]
if missing:
    raise ValueError(f"JSON parsed but these keys are missing: {missing}; present: {sorted(obj)}")
extra = sorted(set(obj) - set(MY_KEYS))
print("PASS: the answer parses and has every key:", MY_KEYS, "| extra keys:", extra or "none")

### Ask for a one-word answer

The cheapest answer is one word. Alert systems ask a yes/no question about each chunk or each candidate clip: the
condition is met or it is not. A one-word answer is quick to generate and simple for a program to read. Compare
this call's time with the reports above.

Look for a single word that agrees with what you saw. If `finish_reason` is `length` and the text starts with
`<think>`, the model began to reason and the 10-token cap cut it off. Raise the cap, or tell the model to answer
without explaining.

In [ ]:
# A yes/no question with a 10-token cap: the pattern alert systems use.
tf_question = ("You are a traffic monitoring system. Does the video show any collision, near-miss, "
               "or stalled vehicle? Answer with only 'yes' or 'no'.")
tf = ask(tf_question, max_tokens=10, temperature=0.2, top_p=0.2)
show(tf, "yes_no")

## 3.7 Turn reasoning on and off

Cosmos 3 Super Reasoner can reason before it answers. Add the reasoning template to the user message, and the
model writes its analysis inside `<think>` tags, then its final answer after `</think>`. Reasoning can help on
complex judgments, such as what happened, which vehicle, and why. It costs tokens and time. The clip, the question
and the route stay the same. The reasoning call also gets room for its trace, `max_tokens=4096`, and the prompt
guide's reasoning settings, `temperature=0.6` and `top_p=0.95`.

**Predict first:** how many times longer will the reasoning call take than the direct one?

The reasoning call takes longer. Let it finish; do not interrupt it and rerun.

Then look for two things. Does the trace cite concrete evidence, such as a vehicle or a lane, before its final
answer? Does the final answer agree with the direct one? Check the cited evidence against the clip. A trace can
read convincingly and still invent a detail.

In [ ]:
# The Cosmos 3 reasoning template. The next cell appends it to the same question.
REASONING_PROMPT = """Answer the question using the following format:

<think>
Your reasoning.
</think>

Write your final answer immediately after the </think> tag."""

question = ("You are a traffic operator watching this camera clip. Is there an incident in this clip "
            "(a vehicle stopped in a live lane, a collision, debris, or pedestrians in the roadway)? "
            "Answer yes or no, then say which vehicle is involved and why, citing the evidence you can see.")

# Without the template: a direct answer.
plain = ask(question, max_tokens=512, temperature=0.4, top_p=0.4)
show(plain, "direct")

In [ ]:
# With the template: the prompt guide's reasoning settings (0.6 and 0.95) and a cap large enough for the trace.
reasoned = ask(f"{question}\n\n{REASONING_PROMPT}", max_tokens=4096, temperature=0.6, top_p=0.95)
text = reasoned["text"]
trace = text.split("</think>", 1)[0].replace("<think>", "").strip() if "</think>" in text else ""
answer = final_answer(text)

print(f"[reasoning] {reasoned['seconds']:.1f} s | finish_reason: {reasoned['finish_reason']} | "
      f"tokens in/out: {reasoned['prompt_tokens']}/{reasoned['completion_tokens']}")
if trace:
    print(f"--- trace ({len(trace.split())} words) ---\n{trace}\n")
else:
    print("No closed <think> block: the trace was cut by max_tokens or the model skipped it. The whole output follows.")
print(f"--- final answer ---\n{answer}\n")

ratio = reasoned["seconds"] / max(plain["seconds"], 1e-6)
print(f"Cost of reasoning: {ratio:.1f}x the time ({reasoned['seconds']:.1f} s vs {plain['seconds']:.1f} s); "
      f"tokens out {reasoned['completion_tokens']} vs {plain['completion_tokens']}. Compare with your prediction.")

Compare the two responses. Without the template the model answers directly. With it, the model works through the
clip inside `<think>` before a short final answer. That can be better grounded on a complex scene, but the trace can
also state evidence that is not in the clip. The price is the extra tokens, so compare the times and the token
counts.

Use reasoning when accuracy on a complex judgment matters. Skip it when speed matters, as with the one-word answer
in section 3.6.

<div class="alert alert-block alert-info">
<b>Tip:</b> with the template, set <code>max_tokens</code> to 4096 so the trace is not cut off before the answer.
The Cosmos 3 prompt guide recommends <code>temperature=0.6</code> and <code>top_p=0.95</code> for reasoning, and
<code>0.7</code> and <code>0.8</code> without it. Pick one setting per application and keep it.
</div>

## 3.8 Find the stored captions

RT-VLM publishes every `generate_captions` caption to Kafka (the message bus) on the topic `mdx-vlm-captions`.
Logstash then indexes the caption text in Elasticsearch. Each registered file gets its own index,
`default_<file id>`. It is separate from the `mdx-raw-*`, `mdx-behavior-*` and `mdx-embed-filtered-*`
indices, which hold the records of RT-CV, Behavior Analytics
and RT-Embed (video search). List the caption indices, then read a few stored captions. The list can hold more than the file you registered. In the
course's test run it showed 9: one for the file of section 3.1, and one for each of the 8 `chat/completions` requests
of sections 3.5 to 3.7, each with one caption.

In [ ]:
r = requests.get(course_setup.BASE + "/es/_cat/indices/default_*?v", timeout=20)
r.raise_for_status()
print(r.text)

In [ ]:
# The caption index is named after the RT-VLM file id, with each "-" replaced by "_".
# Pick it by file id: every registered file has its own index.
index = "default_" + file_id.replace("-", "_")

# Captions can come back before Logstash has indexed them: wait up to 60 s.
started = time.monotonic()
while True:
    r = requests.get(course_setup.BASE + "/es/" + index + "/_search", params={"size": 5}, timeout=30)
    if r.status_code == 200 and r.json().get("hits", {}).get("hits"):
        break
    if r.status_code not in (200, 404):
        r.raise_for_status()
    elapsed = time.monotonic() - started
    if elapsed > 60:
        raise TimeoutError("%s is still empty after %.0f s. If you ran section 3.1 again, for example after a "
                           "kernel restart, it registered a new file with no captions yet: run section 3.2 again. "
                           "Otherwise check Logstash." % (index, elapsed))
    print("waiting for the caption index... %.0f s" % elapsed, flush=True)
    time.sleep(2)

for h in r.json()["hits"]["hits"]:
    meta = h["_source"].get("metadata", {}).get("content_metadata", {})
    print("chunk", meta.get("chunkIdx"), ":", h["_source"]["text"][:110])

The index keeps only the latest caption for each chunk number. A new `generate_captions` call on this file replaces
the stored captions of the chunks it returns. So the captions above come from your latest call: if you ran the cells
in order, the one-sentence answers at the end of section 3.4.

A full-text query matches the words in a caption, not its meaning. A search for "collision" does not find a caption
that says only "collided" or "crash". The next cell searches for "collision" alone, then for the word forms a
caption may use.

**Predict first:** look at the captions above. Which chunks will each search find?

The query searches only the words the model wrote, so it cannot find a detail that no caption described. Notebook 4
searches video embeddings instead, for that reason.

In [ ]:
# A match query finds whole words, so "collision" does not match "collided". Any one of the words is enough.
for words in ("collision", "collision collide collided crash"):
    matching = course_setup.es("POST", "/" + index + "/_search", json={
        "query": {"match": {"text": words}}, "size": 3,
    })
    print(matching["hits"]["total"]["value"], "matching chunks for %r" % words)
    for h in matching["hits"]["hits"]:
        print("-", h["_source"]["text"][:110])

### Optional: find the incident record that a "yes" creates

On this deployment, RT-VLM also looks for "yes" or "true" in each `generate_captions` answer. When it finds one, it
publishes an incident record to Kafka on the topic `mdx-vlm-incidents`, and Logstash indexes it in
`mdx-vlm-incidents-*`. The safety question in section 3.3 asked for yes or no, so it may have created one. This is
the cheapest kind of alert. It is also crude: the check matches the letters anywhere in the text, so "eyes" and
"untrue" trigger it too. The list may also show an index dated `1970-01-01`: a `chat/completions` request carries no
start time, so a yes or true in a chat answer of sections 3.5 to 3.7 lands there.

In [ ]:
r = requests.get(course_setup.BASE + "/es/_cat/indices/mdx-vlm-incidents-*?v", timeout=20)
r.raise_for_status()
print(r.text)

hits = course_setup.es("POST", "/mdx-vlm-incidents-*/_search", json={"size": 1})["hits"]["hits"]
print("fields in one incident record:", sorted(hits[0]["_source"]) if hits else "none yet")

## 3.9 Decide when to run the VLM

<img src="../images/course/caption-cost.png" width="960" alt="Two 24-hour bars for one camera: captioning every ten-second segment fills the first with 8,640 requests, while the second captions only a few example windows that another signal selected, from tracklets or search hits. Below, the cost multiplies out as 8,640 requests times vlm_latency_ms times your cameras, with the takeaway: Multiply it out before you promise continuous captioning.">

Captioning every 10-second chunk of one camera for a day takes 8,640 requests. One request takes about the
`vlm_latency_ms` you saw. Multiply the two to get one camera's GPU time per day, if RT-VLM answered the requests
one at a time. Multiply that by your number of cameras before you promise continuous captioning. The next cell makes a first estimate from your own numbers.

In [ ]:
# A first estimate from section 3.2. Its first chunk covers a full 10 s.
latency_ms = describe[0]["vlm_latency_ms"]
requests_per_day = 24 * 60 * 60 // 10            # one request per 10 s chunk, all day
gpu_seconds = requests_per_day * latency_ms / 1000

print("requests per camera per day at 10 s chunks:", requests_per_day)
print("vlm_latency_ms for one 10 s chunk: %.0f" % latency_ms)
print("GPU time per camera per day, if the requests ran one at a time: %.0f s (%.1f hours)" % (
    gpu_seconds, gpu_seconds / 3600))

### Measure it: one request at a time, then several at once

The estimate above assumes that RT-VLM answers one request at a time. Cameras do not wait for each other, so
requests arrive together. The next cell sends the same caption request 4 times in a row, then 4 times at once, as 4
cameras would, and times both runs. Each request asks for one answer about the whole clip, from 8 frames. The cell
takes about a minute.

**Predict first:** when the 4 requests arrive at once, does each answer take longer than alone? Do all 4 take longer
than 4 in a row?

In [ ]:
# The same request 4 times in a row, then 4 at once (like 4 cameras sending together), timed on the wall clock.
from concurrent.futures import ThreadPoolExecutor

N = 4


def one_request(_=None):
    """One caption of the whole clip from 8 frames; returns (wall seconds, vlm_latency_ms of the answer)."""
    started = time.perf_counter()
    chunks = course_setup.caption(file_id, DESCRIBE, chunk_duration=0, frames=8, use_fps=False)
    return time.perf_counter() - started, chunks[0]["vlm_latency_ms"]


started = time.perf_counter()
in_a_row = [one_request() for _ in range(N)]
row_s = time.perf_counter() - started

started = time.perf_counter()
with ThreadPoolExecutor(max_workers=N) as pool:
    at_once = list(pool.map(one_request, range(N)))
once_s = time.perf_counter() - started

for label, runs, total in (("in a row", in_a_row, row_s), ("at once", at_once, once_s)):
    each = sum(wall for wall, _ in runs) / N
    print(f"{label:9} {N} answers in {total:5.1f} s; each answer took {each:4.1f} s on average")
print(f"\nanswers per minute: in a row {N / row_s * 60:.0f}, at once {N / once_s * 60:.0f}")

**Check your prediction.** Compare the two lines. Sent at once, each answer usually takes a little longer than alone,
because the 4 requests share the GPU. All 4 still finish sooner than 4 in a row, so the answers per minute go up:
RT-VLM processed them together. That is why the first estimate overstates the GPU time, and why you measure with the
number of cameras you plan to run.

The estimate treated the requests as if RT-VLM answered them one at a time. Your measurement shows what happens when
they arrive together: RT-VLM batches them. One GPU then keeps up with more cameras than the first estimate suggests,
though each answer can take longer. Before you size a system, measure latency with as many cameras sending at once
as you plan to run. Use the chunk size, frame count and answer length you will use.

Part 2 runs on its own launch with a smaller model, Cosmos 3 Nano. You fine-tune it for one alert question and
serve it next to RT-CV on two GPUs. The latencies you measured here do not carry over to it.

If an application only needs to look at specific events, let a cheaper signal pick the windows first. RT-CV
tracklets (notebook 2) and RT-Embed search hits (notebook 4) can both pick them. Then caption only those windows.

Also plan for what the system may not see:

- The model can miss an event that falls between two sampled frames. Eight frames spread over a 10-second chunk
  are more than a second apart.
- The detector cannot name every event. TrafficCamNet RT-DETR has no label for a collision or a stopped vehicle
  (notebook 1 listed its labels). Tracklets can find a car that stands still. A collision needs search or the VLM.
- A caption can read well and still be wrong. Check the claims that matter against the footage.

## Check your understanding

1. If you did the optional count in section 3.3: what made the comparison fair, and what would make it a
   measurement?
2. What changes when you change the prompt?
3. When might you caption selected windows instead of a whole stream?
4. Did the direct call and the reasoning call in section 3.7 agree? If two such calls give different yes/no answers
   to the same question, which one do you trust, and what do you do next?

<details>
<summary>Answers</summary>

1. The same source, the same time window and the same object class. It becomes a measurement when you compare
   both numbers with a count you checked in the footage.
2. You change what the model reports, and often the shape of its answer. You still have to check the content and
   parse the format. An answer that parses is not automatically true.
3. When another signal, such as RT-CV tracklets or RT-Embed search hits, can find likely events. Then only those
   events need a detailed description.
4. Agreement is not proof: both calls see the same frames, so they can be wrong together. If they disagree, trust
   neither on its own. Watch the clip and note what is visible. The trace is a claim to check, not evidence. If the
   two keep disagreeing on similar clips, test both on clips whose answer you have checked.

</details>

## Next

Open [4. RT-Embed: Video Search](04-rt-embed-video-search.ipynb). Caption search only finds the words the model
wrote. RT-Embed searches the video itself.

<!-- generated by task1/scripts/course_path.py from part1/course_path.json; edit the JSON, not this cell -->
> **You are done with this step when:** the check cell of the exercise in 3.6 prints `PASS`.
>
> **Next: [4. RT-Embed: Video Search](04-rt-embed-video-search.ipynb)** &middot; Part 1 &middot; about 30 min  
> Search four traffic videos, write queries like captions, score the search against labels, and search one file at a time.
>
> After that: **[Part 1 Assessment](assessment.ipynb)** (about 25 min). On your own, you use VIOS, RT-CV, Kafka, Elasticsearch and RT-VLM on a new clip, search the four traffic videos with RT-Embed, then answer eight questions.
>
> [Back to Home](../00_Home.ipynb)


<center><a href="https://www.nvidia.com/en-us/training/"><img src="../images/nvidia_header.png" width="400"/></a></center>